# Unit 2 — Advanced Visualization & Storytelling (UE24CS342AA9)
## Activity Notebook: Building the Global Development Monitor - PES2UG24CS461

### Scenario

You've just joined the analytics team at the **Global Development Observatory**, an NGO
that briefs policymakers on world development trends. Your director has been burned
before by bad visualizations, and gives you a blunt brief:

1. *"The last analyst showed me a map that made me think Greenland mattered more than
   India. Don't let that happen again."*
2. *"We're about to publish a model predicting life expectancy. If a journalist asks
   'why did the model predict THAT for Norway specifically,' I need an answer, not just
   a feature-importance bar chart for the whole model."*
3. *"Every number in our reports is an estimate. I want our numbers to LOOK like
   estimates, not like certainties."*
4. *"I want one screen I can glance at every morning that tells me what's changing."*

You'll work through the same steps a real analyst would: fix a misleading map, explain
individual model predictions, add honest uncertainty to an estimate and a trend, then
build a small coordinated-views dashboard.

Cells marked **`# TODO`** are for you to complete. Markdown cells marked **Reflection**
are for you to answer in your own words.

## Part 0 — Setup (given)

Run this cell as-is. It loads the real dataset and real country centroids you'll be
working with.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import plotly.express as px
from sklearn.ensemble import RandomForestRegressor
import shap

import ipywidgets as widgets
from ipywidgets import interact, Dropdown

np.random.seed(0)

gapminder = px.data.gapminder()
centroids = pd.read_csv("country_centroids_computed.csv")
gap_latest = gapminder[gapminder.year == 2007].merge(centroids, on="iso_alpha", how="left")
gap_latest = gap_latest.assign(total_gdp=gap_latest["pop"] * gap_latest.gdpPercap)

print(f"Loaded {gapminder.country.nunique()} countries, years {sorted(gapminder.year.unique())}.")
gap_latest[["country", "continent", "year", "lifeExp", "pop", "gdpPercap"]].head()

## Task 1 — Fix the Choropleth Trap (Complaint #1)

**Real-world usage:** The director was misled by a raw-count choropleth that made a
huge, sparsely-populated region look more important than it should.

**Why this technique:** Distinguishing **counts** from **rates** is the single most
important geospatial fix from the lecture — a "count" choropleth is dominated by
population size, not by the thing you actually care about.

**TODO:**
1. Build a choropleth of `total_gdp` (a count-like aggregate) using `px.choropleth`.
2. Build a second choropleth of `gdpPercap` (the rate).
3. Print the top-5 countries by each, and compare.

In [ ]:
# 1. Choropleth of total_gdp (count-like aggregate)
fig_total = px.choropleth(
    gap_latest,
    locations="iso_alpha",
    color="total_gdp",
    hover_name="country",
    color_continuous_scale="Blues",
    title="Total GDP (2007) — count/aggregate framing",
)
fig_total.show()

# 2. Choropleth of gdpPercap (rate)
fig_rate = px.choropleth(
    gap_latest,
    locations="iso_alpha",
    color="gdpPercap",
    hover_name="country",
    color_continuous_scale="Viridis",
    title="GDP per capita (2007) — rate framing",
)
fig_rate.show()

# 3. Top-5 rankings
top_total = gap_latest.nlargest(5, "total_gdp")[["country", "total_gdp"]]
top_rate = gap_latest.nlargest(5, "gdpPercap")[["country", "gdpPercap"]]
print("Top 5 by total GDP:\n", top_total.to_string(index=False))
print("\nTop 5 by GDP per capita:\n", top_rate.to_string(index=False))

**Reflection:** Which countries appear in one top-5 list but not the other? If you had
to pick ONE map to show the director, which would you pick, and why?

_Your answer:_ Populous economies like the United States, China, Japan, India and Germany dominate the total-GDP list, while small wealthy states like Norway, Kuwait, Singapore and Ireland dominate the per-capita list. For the director's question ("who is prosperous?") the **rate map (GDP per capita)** is the honest one — the total-GDP map really just tells you where the population is, which is why India can outrank a much richer per-capita nation. If we cared about "which economies are geopolitically largest" we'd pick total, but for citizen-level prosperity, rate wins.

## Task 2 — Explain One Specific Prediction (Complaint #2)

**Real-world usage:** The director needs to answer "why did the model predict THAT for
Norway specifically" — a global importance bar chart cannot answer a question about one
country.

**Why this technique:** SHAP local importance decomposes ONE prediction into
per-feature, signed contributions — exactly what a journalist question needs.

**TODO:**
1. Fit a `RandomForestRegressor` predicting `lifeExp` from `["gdpPercap", "pop", "year"]`
   on the full `gapminder` dataset.
2. Build a `shap.TreeExplainer` and compute `shap_values` for all rows.
3. Find the row index for Norway, 2007, and plot its 3 SHAP values as a diverging
   horizontal bar chart (blue if positive, red if negative).

In [ ]:
X_model = gapminder[["gdpPercap", "pop", "year"]].values
y_model = gapminder["lifeExp"].values

# 1. Fit random forest
rf = RandomForestRegressor(n_estimators=200, random_state=0)
rf.fit(X_model, y_model)

# 2. SHAP explainer
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_model)

# 3. Norway 2007
row_idx = gapminder.index[(gapminder.country == "Norway") & (gapminder.year == 2007)][0]
vals = shap_values[row_idx]
feature_names = ["gdpPercap", "pop", "year"]

fig, ax = plt.subplots(figsize=(7, 3.5))
colors = ["#3b6ea5" if v >= 0 else "#b5432e" for v in vals]
ax.barh(feature_names, vals, color=colors)
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("SHAP contribution to predicted lifeExp (years)")
ax.set_title(f"Why the model predicted lifeExp for Norway, 2007 (pred = {rf.predict(X_model[[row_idx]])[0]:.1f})")
plt.tight_layout()
plt.show()

**Reflection:** Which feature contributed most POSITIVELY to Norway's predicted life
expectancy, and which (if any) contributed negatively?

_Your answer:_ `gdpPercap` is the strongest positive contributor for Norway (very high income pushes the prediction up); `year` adds a smaller positive push (later years trend healthier), and `pop` contributes little either way. Global importance can't tell you this — it only says which feature matters on average, not which one drove Norway specifically.

## Task 3 — Make an Estimate Look Like an Estimate (Complaint #3)

**Real-world usage:** Every number in the report should visually communicate its own
uncertainty, not look like a fixed fact.

**Why this technique:** Bootstrap confidence intervals + error bars turn a bare point
estimate into an honest range.

**TODO:**
1. Write `bootstrap_ci(values, n_boot=2000)` that resamples `values` with replacement
   `n_boot` times, computes the mean each time, and returns `(mean, ci_low, ci_high)`
   using the 2.5th and 97.5th percentiles of the bootstrap means.
2. Compute a 95% CI for mean `lifeExp` in 2007 for **two** continents of your choice.
3. Plot both as error bars.

In [ ]:
def bootstrap_ci(values, n_boot=2000, seed=0):
    rng = np.random.default_rng(seed)
    values = np.asarray(values)
    boot_means = np.array([rng.choice(values, size=len(values), replace=True).mean()
                           for _ in range(n_boot)])
    return values.mean(), np.percentile(boot_means, 2.5), np.percentile(boot_means, 97.5)

continent_a, continent_b = "Africa", "Europe"
values_a = gap_latest[gap_latest.continent == continent_a].lifeExp.values
values_b = gap_latest[gap_latest.continent == continent_b].lifeExp.values

result_a = bootstrap_ci(values_a)
result_b = bootstrap_ci(values_b)

means = [result_a[0], result_b[0]]
lows = [result_a[0] - result_a[1], result_b[0] - result_b[1]]
highs = [result_a[2] - result_a[0], result_b[2] - result_b[0]]

fig, ax = plt.subplots(figsize=(6, 4))
ax.errorbar([continent_a, continent_b], means,
            yerr=[lows, highs], fmt="o", capsize=8, color="#3b6ea5")
ax.set_ylabel("Mean life expectancy (2007)")
ax.set_title("Bootstrap 95% CI for mean lifeExp")
plt.show()

print(f"{continent_a}: mean={result_a[0]:.2f}, 95% CI=[{result_a[1]:.2f}, {result_a[2]:.2f}]")
print(f"{continent_b}: mean={result_b[0]:.2f}, 95% CI=[{result_b[1]:.2f}, {result_b[2]:.2f}]")

**Reflection:**

_Your answer:_ Africa has the wider CI, mostly because of higher country-to-country variation in life expectancy (large SD), even though the sample size is similar to Europe's. Both n and SD affect CI width, but here SD is the dominant factor.

## Task 4 — Build the Morning Dashboard (Complaint #4)

**Real-world usage:** The director wants one screen to glance at every morning.

**Why this technique:** Coordinated Multiple Views combine complementary chart types
(map = where, trend = how changing, bar = which is biggest) so several monitoring
questions are answered from a single glance.

**TODO:** Build a 1x3 dashboard:
1. A map-style scatter of country centroids, colored by `lifeExp`.
2. A line chart of world-average `lifeExp` by year (use `gapminder.groupby("year")`).
3. A horizontal bar chart of total population by continent in 2007, sorted.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# 1. Map-style scatter of centroids, colored by lifeExp
scat = axes[0].scatter(gap_latest["centroid_lon"], gap_latest["centroid_lat"],
                       c=gap_latest["lifeExp"], cmap="viridis", s=25, alpha=0.8)
axes[0].set_title("Life expectancy by country (2007)")
axes[0].set_xlabel("Longitude"); axes[0].set_ylabel("Latitude")
fig.colorbar(scat, ax=axes[0], label="lifeExp")

# 2. World-average lifeExp trend
world_trend = gapminder.groupby("year").lifeExp.mean()
axes[1].plot(world_trend.index, world_trend.values, "o-", color="#3b6ea5")
axes[1].set_title("World average life expectancy over time")
axes[1].set_xlabel("Year"); axes[1].set_ylabel("Mean lifeExp")

# 3. Population by continent (2007), sorted
pop_by_cont = gap_latest.groupby("continent")["pop"].sum().sort_values()
axes[2].barh(pop_by_cont.index, pop_by_cont.values, color="#b5432e")
axes[2].set_title("Total population by continent (2007)")
axes[2].set_xlabel("Population")

plt.tight_layout()
plt.show()

**Reflection:**

_Your answer:_ I'd keep the map — it answers "where are the exceptions right now?" in one glance, which is the fastest way to spot regional issues that need attention.

## Task 5 — The Decision (No Code)

_Your final recommendation:_ Standardize on **rate** framings (per-capita) for all default maps — raw counts confuse population size with the thing we actually care about. When journalists ask about a specific country, lead with **local** (SHAP) importance, since global importance can't explain any individual prediction. And any **estimated mean** in the dashboard (e.g. continent-average life expectancy) should always ship with a bootstrap CI, because bare point estimates read as certainties.